# Reference uniform-FMM upward pass

This notebook follows the first tree traversal assembled by `dip-fmm`:

```text
particles
    ↓ P2M
leaf multipoles
    ↓ M2M
parent multipoles
    ↓ M2M
root multipole
```

**P2M compresses** the particles within each occupied leaf into Cartesian coefficients about the leaf centre. **M2M changes the expansion centre without moving particles.** After accumulation, a parent multipole represents the union of its child source regions, and the root represents the complete source distribution.

Only the upward pass is demonstrated. M2L interactions, the downward pass, target evaluation, and near-field traversal are intentionally absent.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import cdfmm

positions = np.array([
    [-0.82, -0.70, -0.64], [0.76, -0.58, -0.42],
    [-0.61, 0.69, -0.37], [0.57, 0.73, 0.66],
    [-0.14, 0.22, 0.51], [0.31, -0.19, 0.12],
    [-0.42, 0.08, -0.11], [0.08, 0.49, -0.72],
])
moments = np.array([
    [0.7, -0.2, 0.1], [-0.4, 0.8, 0.3],
    [0.2, 0.1, -0.6], [-0.3, -0.5, 0.9],
    [0.6, 0.4, -0.2], [-0.1, 0.3, 0.5],
    [0.9, -0.7, 0.2], [-0.5, 0.2, -0.4],
])

options = cdfmm.UniformFmmOptions()
options.expansion_order = 5
options.tree.max_level = 2
options.tree.root_centre = cdfmm.Vec3(0.0, 0.0, 0.0)
options.tree.root_half_width = 1.0

fmm = cdfmm.UniformFmm(positions, options)
fmm.upward_pass(moments)
print(f"nodes: {len(fmm.tree.nodes)}, coefficients per node: {len(fmm.root_multipole)}")


## Populated hierarchy

The plot shows only populated nodes to avoid clutter from complete-tree empty boxes. Lines point from each populated child centre to its parent centre. Their direction corresponds to the implemented M2M displacement `d = parent centre - child centre`. Marker size increases towards the root.


In [ ]:
fig = plt.figure(figsize=(8, 7))
axis = fig.add_subplot(projection="3d")
nodes = fmm.tree.nodes
colours = plt.cm.viridis(np.linspace(0.15, 0.9, options.tree.max_level + 1))

axis.scatter(*positions.T, c="black", marker="x", s=45, label="particles")
for level in range(options.tree.max_level, -1, -1):
    populated = [node for node in nodes if node.level == level and node.source_count > 0]
    centres = np.array([[node.centre.x, node.centre.y, node.centre.z] for node in populated])
    axis.scatter(*centres.T, s=45 + 45 * (options.tree.max_level - level),
                 color=colours[level], label=f"level {level} multipoles")
    for node in populated:
        if node.parent >= 0:
            parent = nodes[node.parent]
            child_xyz = np.array([node.centre.x, node.centre.y, node.centre.z])
            parent_xyz = np.array([parent.centre.x, parent.centre.y, parent.centre.z])
            delta = parent_xyz - child_xyz
            axis.quiver(*child_xyz, *delta, color=colours[level], alpha=0.65,
                        arrow_length_ratio=0.12)

axis.set(xlabel="x", ylabel="y", zlabel="z", title="P2M leaves and M2M child-to-parent aggregation")
axis.legend(loc="upper left", fontsize=8)
plt.tight_layout()


## Root equivalence

The hierarchy and a direct P2M call construct the same truncated expansion about the same root centre. They differ only through floating-point accumulation order, so error should be at round-off level—not merely asymptotically small.


In [ ]:
root_centre = np.array([
    fmm.tree.root_centre.x,
    fmm.tree.root_centre.y,
    fmm.tree.root_centre.z,
])
direct_root = cdfmm.p2m_dipole(
    root_centre, positions, moments, options.expansion_order
)
hierarchical_root = fmm.root_multipole
difference = hierarchical_root - direct_root
relative_norm_error = np.linalg.norm(difference) / np.linalg.norm(direct_root)

print(f"max absolute coefficient error: {np.max(np.abs(difference)):.3e}")
print(f"relative norm error:            {relative_norm_error:.3e}")


In [ ]:
indices = cdfmm.multi_indices(options.expansion_order)
degrees = indices.sum(axis=1)
fig, axis = plt.subplots(figsize=(9, 4))
axis.semilogy(np.arange(len(difference)), np.maximum(np.abs(difference), 1e-20), "o")
for degree in range(1, options.expansion_order + 1):
    boundary = np.flatnonzero(degrees == degree)[0] - 0.5
    axis.axvline(boundary, color="0.85", linewidth=0.8)
axis.set(xlabel="coefficient index (MultiIndexSet order)",
         ylabel="absolute difference",
         title="Hierarchical root minus direct-root P2M")
axis.grid(alpha=0.25)
plt.tight_layout()


## Repeated magnetic states

Geometry belongs to setup, while moments belong to evaluation. Calling `upward_pass` again clears every node expansion before applying the new state. The tree and its Morton permutation are reused; this clean separation does not yet add static-geometry translation caches or other optimisation.


In [ ]:
fmm.upward_pass(np.zeros_like(moments))
max_node_coefficient = max(
    np.max(np.abs(fmm.multipole(node.index))) for node in fmm.tree.nodes
)
print(f"largest coefficient after a zero-moment update: {max_node_coefficient:.1f}")
